# Notebook 03 — Feature Engineering
**Project**: Hospital Readmission Prediction  

Good feature engineering is what separates a strong analyst from someone who just runs `.fit()`.  
This notebook encodes domain knowledge about diabetes care into model-ready features.

Features we'll create:
- `age_numeric` — convert age buckets to ordinal numeric
- `prior_utilization_score` — weighted inpatient + emergency + outpatient
- `comorbidity_count` — number of distinct diagnoses
- `medication_changes` — how many meds had dose adjustments
- `insulin_changed` — binary: insulin dose changed?
- `diag1_category` — ICD-9 grouping for primary diagnosis
- `discharge_risk_group` — clinical risk tiers from disposition
- `high_lab_burden` — above 75th percentile for lab procedures

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/diabetic_first_encounter.csv')
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)
print(f'Loaded: {df.shape}')

Loaded: (71518, 51)


## 1. Age — Ordinal Encoding

In [2]:
# Convert '[10-20)' → 15 (midpoint)
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age_numeric'] = df['age'].map(age_map)
print(df[['age', 'age_numeric']].drop_duplicates().sort_values('age_numeric'))

        age  age_numeric
0    [0-10)            5
1   [10-20)           15
2   [20-30)           25
3   [30-40)           35
4   [40-50)           45
5   [50-60)           55
6   [60-70)           65
7   [70-80)           75
8   [80-90)           85
9  [90-100)           95


## 2. Prior Utilization Score

In [3]:
# Clinical rationale: inpatient visits are more predictive than outpatient
# Weight: inpatient × 3, emergency × 2, outpatient × 1
df['prior_utilization_score'] = (
    df['number_inpatient'] * 3 +
    df['number_emergency'] * 2 +
    df['number_outpatient'] * 1
)

# Verify: higher score → higher readmission rate?
score_bins = pd.cut(df['prior_utilization_score'], bins=[0, 1, 3, 6, 100],
                    labels=['Low (0-1)', 'Med (2-3)', 'High (4-6)', 'Very high (7+)'],
                    include_lowest=True)
print(df.groupby(score_bins)['readmitted_30'].mean().map('{:.1%}'.format))

prior_utilization_score
Low (0-1)          7.8%
Med (2-3)         10.4%
High (4-6)        12.8%
Very high (7+)    19.0%
Name: readmitted_30, dtype: object


## 3. Comorbidity Count

In [4]:
# Count non-null, non-missing diagnosis codes across all 3 diagnosis fields
df['comorbidity_count'] = (
    df[['diag_1', 'diag_2', 'diag_3']]
    .apply(lambda col: col.notna() & (col != '?'))
    .sum(axis=1)
)

print('Comorbidity count distribution:')
print(df['comorbidity_count'].value_counts().sort_index())

Comorbidity count distribution:
comorbidity_count
0        1
1      243
2     1041
3    70233
Name: count, dtype: int64


## 4. Medication Features

In [5]:
MEDICATION_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone'
]

# Number of medications with any dose change (Up or Down)
df['medication_changes'] = df[MEDICATION_COLS].isin(['Up', 'Down']).sum(axis=1)

# Binary: insulin specifically changed?
df['insulin_changed'] = df['insulin'].isin(['Up', 'Down']).astype(int)

# Number of distinct medications prescribed (not 'No')
df['num_active_meds'] = df[MEDICATION_COLS].isin(['Steady', 'Up', 'Down']).sum(axis=1)

print(df[['medication_changes', 'insulin_changed', 'num_active_meds']].describe())

       medication_changes  insulin_changed  num_active_meds
count        71518.000000     71518.000000     71518.000000
mean             0.262004         0.202299         1.182709
std              0.475567         0.401717         0.940276
min              0.000000         0.000000         0.000000
25%              0.000000         0.000000         1.000000
50%              0.000000         0.000000         1.000000
75%              0.000000         0.000000         2.000000
max              4.000000         1.000000         6.000000


## 5. Primary Diagnosis Category (ICD-9 Grouping)

In [6]:
def icd9_category(code):
    """
    Map raw ICD-9 code to clinical category.
    Based on standard ICD-9 chapter groupings.
    """
    if pd.isna(code) or code == '?':
        return 'Unknown'
    code = str(code)
    # V and E codes
    if code.startswith('V'):
        return 'Supplementary'
    if code.startswith('E'):
        return 'External cause'
    try:
        num = float(code)
    except ValueError:
        return 'Other'

    if 390 <= num <= 459 or num == 785:
        return 'Circulatory'
    elif 460 <= num <= 519 or num == 786:
        return 'Respiratory'
    elif 520 <= num <= 579 or num == 787:
        return 'Digestive'
    elif num == 250:
        return 'Diabetes'
    elif 800 <= num <= 999:
        return 'Injury'
    elif 710 <= num <= 739:
        return 'Musculoskeletal'
    elif 580 <= num <= 629 or num == 788:
        return 'Genitourinary'
    elif 140 <= num <= 239:
        return 'Neoplasm'
    elif 290 <= num <= 319:
        return 'Mental'
    else:
        return 'Other'

df['diag1_category'] = df['diag_1'].apply(icd9_category)
print(df['diag1_category'].value_counts())

diag1_category
Circulatory        21894
Other              15476
Respiratory         9776
Digestive           6570
Injury              4779
Musculoskeletal     4080
Genitourinary       3514
Neoplasm            2742
Mental              1548
Supplementary        927
Diabetes             200
Unknown               11
External cause         1
Name: count, dtype: int64


## 6. Discharge Risk Group

In [7]:
# Based on EDA findings: group disposition IDs into clinical risk tiers
def discharge_risk(disp_id):
    home_dispositions = [1, 2, 8]          # Home — moderate risk
    snf_dispositions  = [3, 5, 12, 14]     # Skilled nursing / rehab — lower risk (supervised)
    hha_dispositions  = [6, 10]            # Home health agency — lower risk (monitored)
    ama_dispositions  = [7]                # Left against medical advice — high risk
    if disp_id in ama_dispositions:
        return 'High risk (AMA)'
    elif disp_id in home_dispositions:
        return 'Moderate (home)'
    elif disp_id in snf_dispositions:
        return 'Lower (SNF/rehab)'
    elif disp_id in hha_dispositions:
        return 'Lower (home health)'
    else:
        return 'Other'

df['discharge_risk_group'] = df['discharge_disposition_id'].apply(discharge_risk)
print(df.groupby('discharge_risk_group')['readmitted_30'].agg(['mean', 'count']))

                          mean  count
discharge_risk_group                 
High risk (AMA)       0.095355    409
Lower (SNF/rehab)     0.138752   9917
Lower (home health)   0.095118   8295
Moderate (home)       0.071785  45929
Other                 0.113662   6968


## 7. Additional Binary Flags

In [8]:
# High lab burden (above 75th percentile)
lab_p75 = df['num_lab_procedures'].quantile(0.75)
df['high_lab_burden'] = (df['num_lab_procedures'] > lab_p75).astype(int)

# Admitted from ER
df['admitted_from_er'] = (df['admission_source_id'] == 7).astype(int)

# Emergency admission type
df['emergency_admission'] = (df['admission_type_id'] == 1).astype(int)

# HbA1c tested and abnormal
df['a1c_abnormal'] = df['A1Cresult'].isin(['>7', '>8']).astype(int)
df['a1c_tested'] = (df['A1Cresult'] != 'None').astype(int)

print('New binary features created.')

New binary features created.


## 8. Build Final Modeling Dataset

In [9]:
# Columns to drop
DROP_COLS = [
    'encounter_id', 'patient_nbr',   # IDs
    'weight',                         # 97% missing
    'payer_code',                     # 40% missing, not clinical
    'readmitted',                     # raw target — replaced by readmitted_30
    'diag_1', 'diag_2', 'diag_3',    # replaced by diag1_category + comorbidity_count
] + MEDICATION_COLS                   # replaced by engineered medication features

df_model = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Encode categorical columns
CATEGORICAL_COLS = [
    'race', 'gender', 'age', 'medical_specialty',
    'diag1_category', 'discharge_risk_group',
    'max_glu_serum', 'A1Cresult'
]

# Fill remaining missing values
for col in CATEGORICAL_COLS:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna('Unknown').astype(str)

# One-hot encode
df_model = pd.get_dummies(df_model, columns=CATEGORICAL_COLS, drop_first=True, dtype=int)

# Fill any remaining numeric NaN
df_model = df_model.fillna(df_model.median(numeric_only=True))

print(f'Final modeling shape: {df_model.shape}')
print(f'Features: {df_model.shape[1] - 1}')
print(f'Target distribution: {df_model["readmitted_30"].value_counts().to_dict()}')

df_model.to_csv('../data/diabetic_model_ready.csv', index=False)
print('\nSaved: data/diabetic_model_ready.csv ✓')

Final modeling shape: (71518, 135)
Features: 134
Target distribution: {0: 65225, 1: 6293}

Saved: data/diabetic_model_ready.csv ✓
